# Loan Approval Prediction

**Client scenario:** A bank wants to automate its loan approval process by 
predicting whether an applicant's loan should be approved or rejected, based 
on their financial profile (income, credit history, loan amount) and personal 
details (marital status, education, property area).

In [208]:
import pandas as pd
df=pd.read_csv('train.csv')
print("lenght=",len(df))
print(df.head(614))


lenght= 614
      Loan_ID  Gender Married  ... Credit_History Property_Area Loan_Status
0    LP001002    Male      No  ...            1.0         Urban           Y
1    LP001003    Male     Yes  ...            1.0         Rural           N
2    LP001005    Male     Yes  ...            1.0         Urban           Y
3    LP001006    Male     Yes  ...            1.0         Urban           Y
4    LP001008    Male      No  ...            1.0         Urban           Y
..        ...     ...     ...  ...            ...           ...         ...
609  LP002978  Female      No  ...            1.0         Rural           Y
610  LP002979    Male     Yes  ...            1.0         Rural           Y
611  LP002983    Male     Yes  ...            1.0         Urban           Y
612  LP002984    Male     Yes  ...            1.0         Urban           Y
613  LP002990  Female      No  ...            0.0     Semiurban           N

[614 rows x 13 columns]


In [ ]:
baseline_accuracy=(df['Loan_Status']=='Y').mean()
print('B-A',baseline_accuracy)
df['LoanAmount']=df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term']=df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())
df['Credit_History']=df['Credit_History'].fillna(df['Credit_History'].median())
df['Gender']=df['Gender'].fillna(df['Gender'].mode()[0])
df['Married']=df['Married'].fillna(df['Married'].mode()[0])
df['Dependents']=df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed']=df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
print(df.isnull().sum())

B-A 0.6872964169381107


In [213]:
df['Gender']=df['Gender'].map({'Male':1 ,'Female':0})
df['Married']=df['Married'].map({'Yes':1 ,'No':0})
df['Education']=df['Education'].map({'Graduate':1 ,'Not Graduate':0})
df['Self_Employed']=df['Self_Employed'].map({'Yes':1 ,'No':0})
df['Loan_Status']=df['Loan_Status'].map({'Y':1 ,'N':0})
df['Dependents']=df['Dependents'].replace('3+','3')
df['Dependents']=pd.to_numeric(df['Dependents'])


In [214]:
df['Property_Area'] = df['Property_Area'].map({'Urban': 0, 'Semiurban': 1, 'Rural': 2})

In [215]:
print(df.columns.tolist())

['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


In [216]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
x= df[['Credit_History', 'ApplicantIncome', 'LoanAmount', 'Gender', 'Married', 'Education', 'Property_Area']]
y = df['Loan_Status']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
model=DecisionTreeClassifier(random_state=42,max_depth=8)
model.fit(x_train,y_train)
prediction=model.predict(x_test)
model_accuracy=accuracy_score(y_test,prediction)
print('model_accuracy:',model_accuracy)
print('baseline_accuracy:',baseline_accuracy)
print('improvemnt over baqseline:',model_accuracy-baseline_accuracy)

model_accuracy: 0.7398373983739838
baseline_accuracy: 0.6872964169381107
improvemnt over baqseline: 0.052540981435873046


In [217]:
results=x_test.copy()
results['actual']=y_test
results['predicted']=prediction

wrong_prediction=results[results['actual'] != results['predicted']]
print('wrong_prediction length',len(wrong_prediction))
print(wrong_prediction.head())

wrong_prediction length 32
     Credit_History  ApplicantIncome  ...  actual  predicted
340             1.0             2647  ...       0          1
77              1.0             1000  ...       0          1
209             1.0             3418  ...       0          1
610             1.0             4106  ...       1          0
148             1.0            10000  ...       0          1

[5 rows x 9 columns]


In [218]:
importances = pd.Series(model.feature_importances_, index=x.columns)
print(importances.sort_values(ascending=False))

Credit_History     0.453013
LoanAmount         0.270602
ApplicantIncome    0.184136
Gender             0.036437
Property_Area      0.032743
Married            0.019592
Education          0.003477
dtype: float64


In [219]:
print("Model accuracy:", model_accuracy)


Model accuracy: 0.7398373983739838


In [220]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


In [221]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
baseline=y_test.value_counts(normalize=True).max()
tree=DecisionTreeClassifier(max_depth=4,random_state=42).fit(x_train,y_train)
tree_accuracy=accuracy_score(y_test,tree.predict(x_test))
print('baseline',baseline)                             
print('tree accuracy:',tree_accuracy)                             

baseline 0.6504065040650406
tree accuracy: 0.7560975609756098


In [222]:
knn=KNeighborsClassifier(n_neighbors=3).fit(x_train,y_train)
unscaled_knn=accuracy_score(y_test,knn.predict(x_test))
print('unscaled_knn',unscaled_knn)



unscaled_knn 0.5853658536585366


In [223]:
scaler=StandardScaler()
x_train_s=scaler.fit_transform(x_train)
x_test_s=scaler.transform(x_test)
knn_s=KNeighborsClassifier(n_neighbors=5).fit(x_train_s,y_train)
scaled_knn=accuracy_score(y_test,knn_s.predict(x_test_s))
print('scaled knn',scaled_knn)


scaled knn 0.7886178861788617


In [224]:
for k in [1,3,5,7,9,14]:
    m=KNeighborsClassifier(n_neighbors=k).fit(x_train_s,y_train)
    print(k,round(m.score(x_train_s,y_train),3),round(m.score(x_test_s,y_test)))

1 1.0 1
3 0.859 1
5 0.837 1
7 0.817 1
9 0.809 1
14 0.811 1


In [225]:
import pandas as pd
data={
    'model':['baseline','DecisionTree','scaled_knn','unscaled_knn'],
    'Accuracy':[baseline,tree_accuracy,unscaled_knn,scaled_knn]
}
df=pd.DataFrame(data)
print(df)

          model  Accuracy
0      baseline  0.650407
1  DecisionTree  0.756098
2    scaled_knn  0.585366
3  unscaled_knn  0.788618


In [ ]:
from sklearn.metrics import confusion_matrix
print('connfusion-matrix of tree',confusion_matrix(y_test,tree.predict(x_test)))
print('connfusion-matrix of knn_s',confusion_matrix(y_test,tree.predict(x_test_s)))

connfusion-matrix of tree [[16 27]
 [ 3 77]]
connfusion-matrix of knn_s [[43  0]
 [80  0]]


c:\Users\Qadri Laptop\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


## Summary for Client

Credit History is by far the strongest predictor of loan approval, accounting 
for over 70% of the model's decision-making at max_depth=4. Loan Amount and 
Applicant Income also play a meaningful role, while Gender and Education had 
little to no impact on the model's decisions.

The model achieves **~75% accuracy**, an improvement of roughly 7 percentage 
points over the baseline (68.7%) of always predicting "approved."

We tested tree depths from unrestricted to depth=4 and depth=8. An unrestricted 
tree overfit heavily, memorizing the training data but performing worse on new 
applicants. Depth=4 gave the best balance between accuracy and generalization, 
and was selected as the final model.

**Recommendation:** Prioritize verifying an applicant's credit history and 
requested loan amount relative to their income, as these are the strongest 
signals for approval likelihood.